In [1]:
import pandas as pd
import numpy as np
import re
from rapidfuzz import process, fuzz
import json
import os
from tqdm.auto import tqdm

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_colwidth', None)

## 1. Load Data
- Unmatched classes
- Big recipe dataset
- IFCT 2017 dataset
- OFF Products (for backup)

We will perform strict matching to find recipes for missing classes, and also audit the main dataset for potential mismatches.

In [2]:
# 1. Unmatched Classes
df_unmatched = pd.read_csv('../Dataset/unmatched_image_classes.csv')
classes_to_fix = df_unmatched['class'].tolist()

# 2. Big Recipe Dataset
df_recipes = pd.read_csv('../Dataset/raw/Cleaned_Indian_Food_Dataset.csv')

# 3. IFCT 2017 Dataset
df_ifct = pd.read_csv('../Dataset/ifct2017_compositions.csv')

# 4. OFF Products (Cleaned) - IF AVAILABLE, otherwise use raw and clean lightly
try:
    df_off = pd.read_csv('../Dataset/processed/OFF_Indian_Products_Clean.csv')
except FileNotFoundError:
    print("Cleaned OFF dataset not found, skipping for now.")
    df_off = pd.DataFrame()

print(f"Classes to fix: {len(classes_to_fix)}")
print(f"Total recipes available to search: {len(df_recipes)}")
print(f"IFCT items: {len(df_ifct)}")

Cleaned OFF dataset not found, skipping for now.
Classes to fix: 148
Total recipes available to search: 5938
IFCT items: 542


## 2. Match Classes to Recipes
Find the best matching recipe in the big dataset for each missing image class.

In [12]:
recipe_names = df_recipes['TranslatedRecipeName'].tolist()

def verify_ingredients(image_class, ingredients_text):
    """
    Simple heuristic: Check if key words from the image class (like 'chicken', 'paneer')
    appear in the recipe ingredients.
    """
    # Key ingredients to check for specific classes
    # Add more rules as needed
    class_lower = image_class.lower()
    ingredients_lower = str(ingredients_text).lower()
    
    keywords = {
        'chicken': 'chicken',
        'mutton': 'mutton',
        'fish': 'fish',
        'egg': 'egg',
        'paneer': 'paneer',
        'dal': ['dal', 'lentil', 'gram'],
        'rice': 'rice',
        'bread': ['flour', 'wheat', 'maida'],
        'cake': ['flour', 'sugar', 'baking'],
        'pizza': ['cheese', 'flour', 'dough'],
        'burger': ['bun', 'bread', 'patty'],
    }
    
    for key, values in keywords.items():
        if key in class_lower:
            if isinstance(values, list):
                if not any(v in ingredients_lower for v in values):
                    return False, f"Missing key ingredient: {key}"
            else:
                if values not in ingredients_lower:
                    return False, f"Missing key ingredient: {key}"
            return True, "Verified"
            
    return True, "No specific keyword rule"

matches = []
skipped_count = 0

print("[INFO] Starting strict matching process...")

for cls in tqdm(classes_to_fix):
    # Cleaning the class name slightly for better matching
    clean_cls = cls.replace('_', ' ').title()
    
    # Fuzzy match - Get top 3 to verify best logic
    # extract returns list of (match, score, index)
    candidates = process.extract(clean_cls, recipe_names, scorer=fuzz.token_set_ratio, limit=3)
    
    best_candidate = None
    best_reason = ""
    
    # Try to find a high-quality match among candidates
    for name, score, idx in candidates:
        if score < 85: # Stricter threshold
            continue
            
        recipe_row = df_recipes.iloc[idx]
        ingredients = recipe_row['Cleaned-Ingredients']
        
        # Verify ingredients
        is_valid, reason = verify_ingredients(clean_cls, ingredients)
        
        if is_valid:
            best_candidate = (name, score, recipe_row)
            best_reason = "Score > 85 and Ingredients Verified"
            break # Stop at first good match
        else:
            # If high score but ingredients don't match, we might skip this candidate
            # But let's log it?
            pass

    if best_candidate:
        name, score, recipe_row = best_candidate
        matches.append({
            'image_class': cls,
            'matched_recipe': name,
            'match_score': score,
            'match_reason': best_reason,
            'ingredients': recipe_row['Cleaned-Ingredients'],
            'instructions': recipe_row['TranslatedInstructions']
        })
    else:
        skipped_count += 1
        # Optional: Add a "No Match" entry if you want to track what's missing
        # matches.append({
        #     'image_class': cls,
        #     'matched_recipe': None,
        #     'match_score': 0,
        #     'match_reason': "No valid match found > 85 score",
        #     'ingredients': "",
        #     'instructions': ""
        # })

df_matches = pd.DataFrame(matches)

print(f"\n[OK] Found {len(df_matches)} valid matches.")
print(f"[SKIP] Skipped {skipped_count} classes due to low score or ingredient mismatch.")

if not df_matches.empty:
    print(f"Average Match Score: {df_matches['match_score'].mean():.2f}")
    
# Display sample of what was found
df_matches.head(10)

[INFO] Starting strict matching process...


  0%|          | 0/148 [00:00<?, ?it/s]


[OK] Found 94 valid matches.
[SKIP] Skipped 54 classes due to low score or ingredient mismatch.
Average Match Score: 100.00


,image_class,matched_recipe,match_score,match_reason,ingredients,instructions
0,achar,Singhade Ka Achar Recipe (Water Chestnut Pickle),100.0,Score > 85 and Ingredients Verified,"salt,fennel seeds (saunf),red chilli powder,mustard powder,sugar,vinegar,singhade (water chestnut),turmeric powder,sunflower oil","To begin making the Singhade Ka Achaar recipe, peel the pointed sides, wash and cut into two.Boil it with a glass of water for 10 to 12 minutes in a saucepan, cool and keep in strainer.\r\nLay it in an open space for 1 hour now.\r\nTake a large bowl and mix all the ingredients including the boiled singhade properly.Keep the jar in sun light for 2 days and it's ready to serve.Serve Singhade Ka Achaar along with Pudina Tawa Paratha or Nutty Aloo Paratha for breakfast on a weekend morning.\r\n"
1,aloo gobi,Tandoori Aloo Gobi Kathi Roll Recipe - Spicy Aloo Gobi Frankie,100.0,Score > 85 and Ingredients Verified,"salt,wheat flour,kashmiri red chilli powder,cauliflower (gobi),chaat masala powder,curd,onion,cumin seeds (jeera),lemon,baby potatoes,green bell pepper (capsicum),turmeric powder,green chillies,sunflower oil","To begin making the Aloo Gobi Kathi Roll, we have to first cook the aloo and gobi.\r\nInto a pressure cooker, add the potatoes along with 1/2 cup of water and pressure cook for 3 to 4 whistles and turn off the heat.\r\nAllow the pressure to release naturally.Peel the skin of the potatoes and keep aside.If you have a steamer, steam the cauliflower florets on high heat for 3 to 4 minutes until just about cooked, but its still firm.\r\nYou can also pan roast the cauliflower with a little salt until it has lightly cooked.In a bowl add curd 1/2 cup yogurt along with chaat masala, red chilli powder, turmeric powder and salt.\r\nAdd the baby potatoes and cauliflower florets into the curd mixture and coat the aloo gobi evenly using a spoon.\r\nCheck for seasoning and leave aside for 15 minutes to marinate.\r\n Now insert the baby potatoes and cauliflowers into a sticks or skewers for roasting on a skillet.\r\nPlace a skillet on medium heat and add a teaspoon of oil.\r\nAdd cumin seeds and let them crackle.\r\nPlace the skewered aloo gobi on the skillet and roast it until you see a few brown spots on the vegetables and see it getting roasted.Roast all the skewers on a low to medium heat and keep aside.\r\n*Making the paratha for the Kati RollsIn a bowl mix whole wheat flour, salt and one teaspoon of oil and mix well.\r\nAdd little water at a time and bring the flour together to form a smooth dough.\r\nCover with a muslin cloth and keep aside for 10 minutes.\r\nMake equal portions of the dough and roll it into balls.\r\nOn a clean surface dust some flour and flatten the ball of dough on it.\r\nRoll out the dough into a paratha.\r\nPlace a roti tawa on medium heat and cook all the parathas with ghee or oil.\r\nKeep the cooked parathas aside.\r\nIn a medium bowl, mix together sliced onions and capsicum slices along with lemon juice, salt and green chillies.*Assembling the Kathi RollsPlace a paratha on foil paper.\r\nPlace a small portion of the onion and capsicum slices.\r\nNext place a skewer of roasted tandoori aloo gobi in the centre.\r\nPull the skewer out and discard.\r\nRoll the parathas along with the foil and prepare the rest of the Tandoori Aloo Gobi Kathi rolls the similar way.\r\nServe Tandoori Aloo Gobi Kathi Rolls along with a Dhaniya Pudina Chutney and a cup of Masala Chai for a tea time meal or pack these Aloo Gobi Kathi Rolls for lunch along with a cup of curd.\r\n"
2,aloo methi,Methi & Pudina Aloo Sabzi Recipe,100.0,Score > 85 and Ingredients Verified,"tomato,salt,ginger,coriander powder,methi leaves (fenugreek leaves),red chilli powder,onion,cumin seeds (jeera),mint leaves (pudina),potato (aloo),turmeric powder,garam masala powder,sunflower oil","To begin making the Aloo Methi & Pudina Sabzi Recipe in two stage process.\r\nWe will first roast the potatoes and keep them aside.\r\nWe will roast the onions, toma

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# OPTIONAL: Audit Main Recipe Dataset for Mismatches
# ─────────────────────────────────────────────────────────────────────────────
print("[INFO] Auditing Main Recipe Dataset for Potential Mismatches...")

mismatches = []
rows_to_check = df_recipes.sample(min(len(df_recipes), 2000), random_state=42) # Check a sample first

for idx, row in tqdm(rows_to_check.iterrows(), total=len(rows_to_check)):
    name = str(row['TranslatedRecipeName']).lower()
    ingredients = str(row['Cleaned-Ingredients']).lower()
    instructions = str(row['TranslatedInstructions']).lower()
    
    issues = []
    
    # Check 1: Vegetarian Mismatches
    # Name implies veg, but ingredients/instructions imply non-veg
    non_veg_terms = ['chicken', 'mutton', 'fish', 'prawn', 'shrimp', 'crab', 'egg', 'beef', 'pork', 'meat']
    if 'veg' in name and not 'non-veg' in name and not 'egg' in name:
        found_terms = [term for term in non_veg_terms if term in ingredients]
        # Filter out common false positives like "eggplant" or "chickpea" if checking just substrings
        # Better to check for word boundaries, but simple check for now:
        real_found = []
        for term in found_terms:
            # Check whole word
            if re.search(rf'\b{term}\b', ingredients):
                # special case: 'egg' in 'eggplant' handled by \b, but 'eggless' might trip
                if term == 'egg' and 'eggless' in ingredients: continue
                real_found.append(term)
        
        if real_found:
             issues.append(f"Name suggests Veg, but ingredients contain: {real_found}")

    # Check 2: Non-Vegetarian Mismatches
    # Name implies specific meat (e.g., Chicken), but ingredients don't have it
    meat_map = {
        'chicken': 'chicken',
        'mutton': 'mutton', 
        'fish': 'fish',
        'prawn': ['prawn', 'shrimp'],
        'egg': 'egg'
    }
    
    for meat_key, meat_vals in meat_map.items():
        # If title says "Chicken", ingredients MUST have "Chicken"
        if meat_key in name and 'mock' not in name and 'soy' not in name:
             if isinstance(meat_vals, list):
                 if not any(val in ingredients for val in meat_vals):
                     issues.append(f"Name says '{meat_key}', but ingredients missing {meat_vals}")
             else:
                 if meat_vals not in ingredients:
                     # Double check instructions just in case
                     if meat_vals not in instructions:
                        issues.append(f"Name says '{meat_key}', but ingredients missing '{meat_vals}'")

    # Check 3: Suspiciously Short Ingredients
    if len(row['Cleaned-Ingredients'].split(',')) < 2:
        issues.append("Suspiciously few ingredients (< 2)")

    if issues:
        mismatches.append({
            'Index': idx,
            'Recipe Name': row['TranslatedRecipeName'],
            'Issues': "; ".join(issues),
            'Ingredients': row['Cleaned-Ingredients']
        })

df_audit = pd.DataFrame(mismatches)

if not df_audit.empty:
    print(f"\n[WARN] Found {len(df_audit)} potential mismatches in sample.")
    print("Top issues found:")
    display(df_audit.head(10))
else:
    print("\n[OK] No obvious mismatches found in sample.")

[INFO] Auditing Main Recipe Dataset for Potential Mismatches...


  0%|          | 0/2000 [00:00<?, ?it/s]


[WARN] Found 13 potential mismatches in sample.
Top issues found:


,Index,Recipe Name,Issues,Ingredients
0,5697,Spicy & Tangy Goan Chicken Stew Recipe With Vegetables,"Name suggests Veg, but ingredients contain: ['chicken']","tomato,chicken (preferably bs),cumin powder (jeera),ginger,salt,potato (aloo),green peas (matar),cloves garlic,carrot,water,red chilli powder,onion,gingelly oil,lemon,tamarind paste,black pepper powder,coriander powder,green chillies"
1,3399,Chinese Dragon Chicken With Vegetables Recipe,"Name suggests Veg, but ingredients contain: ['chicken']","yellow bell pepper (capsicum),cashew nuts,red bell pepper (capsicum),corn flour,gram baby corn cubes,gram button mushrooms,red chilli flakes,chicken bless,carrot,ginger garlic paste,tomato ketchup,onion,red chilli,green bell pepper (capsicum),eggs,red chilli sauce,sunflower oil"
2,3254,Veggie Rava Idli (Recipe In Hindi),"Name says 'egg', but ingredients missing 'egg'","salt,ginger,yogurt,red chilli flakes,carrot,mustard,semolina,corianderhua,chana dal,curry leaves,oil,water,sweet corn,ino fruit salt,green chillies,green peas"
3,2771,Orange Zested Shortbread Cookie Recipe (Egg Free/Eggless),"Name says 'egg', but ingredients missing 'egg'","salt,orange,dark chocolate semi sweet,vanilla,caster sugar,hazelnut (or cashew nuts),flour (maida),butter"
4,5637,Eggless Cranberry Pistachio Biscotti Recipe,"Name says 'egg', but ingredients missing 'egg'","baking powder,flour (maida) plus,salt,sugar,vanilla,cranberries dried,pistachios unsalted,sunflower oil"
5,1978,Nankhatai Recipe - A Spiced Eggless Indian Cookie,"Name says 'egg', but ingredients missing 'egg'","salt,baking powder,ghee,gram flour (besan),curd,caster sugar,cardamom powder (elaichi),sooji (semolina rava),flour (maida)"
6,1425,Macha Mahura Recipe - Odia Style Fish Curry With Vegetables,"Name suggests Veg, but ingredients contain: ['fish']","salt,elephant yam (suransenairatalu),parangikai pumpkin,fennel seeds (saunf),plantain stem,black peppercorns,water,ginger garlic paste,coriander (dhania) seeds,colocasia root (arbi),badi (wadi),cumin seeds (jeera),cinnamon stick (dalchini),panch phoran masala,turmeric powder,onion,rohu fish,cloves (laung),kala chana (brown chickpeas),sunflower oil"
7,4292,Healthy Lemon Cupcakes Recipe (Eggless),"Name says 'egg', but ingredients missing 'egg'","baking powder,wheat flour,water,baking soda,lemon,caster sugar,raisins black,turmeric powder,sunflower oil"
8,3095,Delicious Italian Chicken Cacciatore Recipe with Mushrooms & Olives - (Poultry & Vegetable Hunter's Stew),"Name suggests Veg, but ingredients contain: ['chicken']","tomato,rosemary,basil leaves,cloves garlic,del monte black olives,virgin olive oil,parsley leaves,onion,dry white wine,button mushrooms,chicken breasts"
9,744,Grilled Chicken With Vegetables Recipe,"Name suggests Veg, but ingredients contain: ['chicken']","salt,red bell pepper (capsicum),mixed herbs (dried),dried basil leaves,black peppercorns,virgin olive oil,ginger garlic paste,red chilli flakes,baby corn,onion,garlic,green bell pepper (capsicum),broccoli florets,chicken breasts,cherry tomato"


In [5]:
# Display all matches
with pd.option_context('display.max_rows', None):
    display(df_matches)
    

,image_class,matched_recipe,match_score,ingredients,instructions
0,achar,Singhade Ka Achar Recipe (Water Chestnut Pickle),100.000000,"salt,fennel seeds (saunf),red chilli powder,mustard powder,sugar,vinegar,singhade (water chestnut),turmeric powder,sunflower oil","To begin making the Singhade Ka Achaar recipe, peel the pointed sides, wash and cut into two.Boil it with a glass of water for 10 to 12 minutes in a saucepan, cool and keep in strainer.\r\nLay it in an open space for 1 hour now.\r\nTake a large bowl and mix all the ingredients including the boiled singhade properly.Keep the jar in sun light for 2 days and it's ready to serve.Serve Singhade Ka Achaar along with Pudina Tawa Paratha or Nutty Aloo Paratha for breakfast on a weekend morning.\r\n"
1,adhirasam,Maharashtrian Kadhi Recipe,40.000000,"hung curd (greek yogurt),ginger,green chilli,mustard seeds,asafoetida (hing),gram flour (besan),curry leaves,sugar,coriander (dhania) leaves,cumin seeds (jeera)","To begin making the Maharashtrian Kadhi Recipe, first mix the yogurt and besan in a bowl.\r\nAdd 1 cup water and whisk it to a smooth mixture without any lumps.Heat a sauce pan with oil, add mustard seeds and cumin seeds, leave it to splutter for 10 seconds.Add asafoetida, green chilli, ginger and curry leaves and fry them for 10 more seconds till the aroma is released.At this stage you can add the yogurt and besan mixture and keep at a low heat and leave it to boil.\r\nKeep whisking till it thickens.Add salt and sugar and adjust it according to your taste.\r\nSwitch off the heat.Once the kadhi is ready, garnish the kadhi with some chopped fresh coriander leaves.Serve the Maharashtrian Kadhi Recipe along with Gajarachi Koshimbir, Steamed Rice and a roasted Papad for a delicious weekday meal."
2,aloo gobi,Tandoori Aloo Gobi Kathi Roll Recipe - Spicy Aloo Gobi Frankie,100.000000,"salt,wheat flour,kashmiri red chilli powder,cauliflower (gobi),chaat masala powder,curd,onion,cumin seeds (jeera),lemon,baby potatoes,green bell pepper (capsicum),turmeric powder,green chillies,sunflower oil","To begin making the Aloo Gobi Kathi Roll, we have to first cook the aloo and gobi.\r\nInto a pressure cooker, add the potatoes along with 1/2 cup of water and pressure cook for 3 to 4 whistles and turn off the heat.\r\nAllow the pressure to release naturally.Peel the skin of the potatoes and keep aside.If you have a steamer, steam the cauliflower florets on high heat for 3 to 4 minutes until just about cooked, but its still firm.\r\nYou can also pan roast the cauliflower with a little salt until it has lightly cooked.In a bowl add curd 1/2 cup yogurt along with chaat masala, red chilli powder, turmeric powder and salt.\r\nAdd the baby potatoes and cauliflower florets into the curd mixture and coat the aloo gobi evenly using a spoon.\r\nCheck for seasoning and leave aside for 15 minutes to marinate.\r\n Now insert the baby potatoes and cauliflowers into a sticks or skewers for roasting on a skillet.\r\nPlace a skillet on medium heat and add a teaspoon of oil.\r\nAdd cumin seeds and let them crackle.\r\nPlace the skewered aloo gobi on the skillet and roast it until you see a few brown spots on the vegetables and see it getting roasted.Roast all the skewers on a low to medium heat and keep aside.\r\n*Making the paratha for the Kati RollsIn a bowl mix whole wheat flour, salt and one teaspoon of oil and mix well.\r\nAdd little water at a time and bring the flour together to form a smooth dough.\r\nCover with a muslin cloth and keep aside for 10 minutes.\r\nMake equal portions of the dough and roll it into balls.\r\nOn a clean surface dust some flour and flatten the ball of dough on it.\r\nRoll out the dough into a paratha.\r\nPlace a roti tawa on medium heat and cook all the parathas with ghee or oil.\r\nKeep the cooked parathas aside.\r\nIn a medium bowl, mix together sliced onions and capsicum slices along with lemon juice, salt and green chillies.*Assembling the Kathi RollsPlace a paratha on f

## 3. Ingredient Parser
Implement the regex-based parser to extract quantity, unit, and name.

In [9]:
# ─── Unit densities (grams per 1 of that unit) ───────────────────────────────
UNIT_DENSITY = {
    'cup': 240, 'tablespoon': 15, 'teaspoon': 5,
    'gram': 1, 'kg': 1000, 'liter': 1000, 'ml': 1,
    'pinch': 0.5, 'piece': 80, 'medium': 80,
    'large': 150, 'small': 50, 'handful': 30,
    'bunch': 100, 'clove': 5, 'inch': 5,
    'sprig': 2, 'leaves': 2, 'packet': 200, 'slice': 30
}

# Abbreviation → full unit name
UNIT_ABBREVS = {
    'tsp': 'teaspoon', 'tsps': 'teaspoon',
    'tbsp': 'tablespoon', 'tbsps': 'tablespoon', 'tbs': 'tablespoon', 'tb': 'tablespoon',
    'g': 'gram', 'gm': 'gram', 'gms': 'gram',
    'kg': 'kg', 'kgs': 'kg',
    'ml': 'ml', 'mls': 'ml', 'cc': 'ml',
    'l': 'liter', 'ltr': 'liter', 'ltrs': 'liter', 'litre': 'liter', 'litres': 'liter',
    'pcs': 'piece', 'pc': 'piece', 'nos': 'piece',
}

# Per-ingredient density for volume units (g/cup equivalent)
INGREDIENT_DENSITY = {
    'rice': 185, 'dal': 200, 'lentil': 200,
    'flour': 120, 'wheat flour': 120, 'maida': 125, 'besan': 92,
    'sugar': 200, 'salt': 288,
    'oil': 218, 'ghee': 227, 'butter': 227,
    'water': 240, 'milk': 244, 'yogurt': 245, 'curd': 245,
    'onion': 150, 'tomato': 180, 'potato': 150, 'spinach': 30
}

WORDS_TO_STRIP = [
    'finely chopped', 'roughly chopped', 'thinly sliced',
    'chopped', 'sliced', 'diced', 'cubed', 'grated', 'minced',
    'crushed', 'ground', 'washed', 'cleaned', 'peeled',
    'roasted', 'toasted', 'soaked', 'cooked', 'boiled',
    'fresh', 'dried', 'slit', 'about', 'optional',
    'for garnish', 'garnishing', 'for frying', 'for cooking',
    'to taste', 'as needed'
]

# Sort units longest-first so 'tablespoon' is matched before 'table'
UNITS_SORTED = sorted(UNIT_DENSITY.keys(), key=len, reverse=True)
# Sort abbreviations longest-first to avoid ambiguous prefix matches
ABBREVS_SORTED = sorted(UNIT_ABBREVS.keys(), key=len, reverse=True)


def parse_qty(text: str):
    """
    Extract a leading numeric quantity.
    Handles: mixed fractions '1 1/2', plain fractions '1/2',
             ranges '3-4' (→ midpoint), decimals, integers.
    Returns (float, remaining_string).
    """
    text = text.strip()

    # Mixed fraction: "1 1/2"
    m = re.match(r'^(\d+)\s+(\d+)/(\d+)', text)
    if m:
        return int(m.group(1)) + int(m.group(2)) / int(m.group(3)), text[m.end():].strip()

    # Simple fraction: "3/4"
    m = re.match(r'^(\d+)/(\d+)', text)
    if m:
        return int(m.group(1)) / int(m.group(2)), text[m.end():].strip()

    # Range: "3-4" → midpoint
    m = re.match(r'^(\d+\.?\d*)\s*[-\u2013]\s*(\d+\.?\d*)', text)
    if m:
        return (float(m.group(1)) + float(m.group(2))) / 2, text[m.end():].strip()

    # Decimal / integer
    m = re.match(r'^(\d+\.?\d*)', text)
    if m:
        return float(m.group(1)), text[m.end():].strip()

    return 1.0, text  # implied quantity


def _expand_abbreviation(text: str) -> str:
    """Expand any leading unit abbreviation to its full name."""
    for abbr in ABBREVS_SORTED:
        pat = rf'^{re.escape(abbr)}s?\b'
        if re.match(pat, text):
            return re.sub(pat, UNIT_ABBREVS[abbr], text, count=1).strip()
    return text


def parse_ingredient(line: str) -> dict:
    """Parse one ingredient line → {qty, unit, name, raw}."""
    raw = line
    line = line.lower().strip()

    # Remove parenthetical translations: "hing (asafoetida)"
    line = re.sub(r'\(.*?\)', '', line).strip()

    # "to taste" / "as needed" → tiny default amount
    if re.search(r'\bto taste\b|\bas needed\b', line):
        name = re.sub(r'\bto taste\b|\bas needed\b', '', line).strip(' ,')
        return {'qty': 0.25, 'unit': 'teaspoon', 'name': name, 'raw': raw}

    # "a pinch of …"
    if re.search(r'\ba\s+pinch\b', line):
        name = re.sub(r'\ba\s+pinch\s+of\b|\ba\s+pinch\b', '', line).strip(' ,')
        return {'qty': 0.125, 'unit': 'teaspoon', 'name': name, 'raw': raw}

    # Extract quantity
    qty, line = parse_qty(line)

    # Expand abbreviation before matching full unit names
    line = _expand_abbreviation(line)

    # Extract unit (longest-first)
    unit = 'piece'
    for u in UNITS_SORTED:
        pat = rf'^{re.escape(u)}s?\b'
        if re.match(pat, line):
            unit = u
            line = re.sub(pat, '', line, count=1).strip()
            if line.startswith('of '):
                line = line[3:]
            break

    # Clean descriptor words from name
    name = line.strip(' ,.')
    for w in sorted(WORDS_TO_STRIP, key=len, reverse=True):
        name = re.sub(rf'\b{re.escape(w)}\b', '', name, flags=re.IGNORECASE)
    name = re.sub(r'\s+', ' ', name).strip(' ,.')

    return {'qty': qty, 'unit': unit, 'name': name, 'raw': raw}


def convert_to_grams(parsed: dict) -> float:
    """Convert a parsed ingredient dict to grams using density tables."""
    unit = parsed['unit']
    qty  = parsed['qty']
    name = parsed['name']
    base = UNIT_DENSITY.get(unit, 1)

    # Adjust volume units by ingredient-specific density
    if unit in ('cup', 'tablespoon', 'teaspoon'):
        for key, density in INGREDIENT_DENSITY.items():
            if key in name:
                return qty * UNIT_DENSITY[unit] * (density / 240.0)
    return qty * base


# ─── Sanity check ─────────────────────────────────────────────────────────────
tests = [
    "2 cups basmati rice",
    "1/2 tsp turmeric",
    "1 1/2 tbsp oil",
    "3-4 green chillies",
    "salt to taste",
    "1 medium onion, finely chopped",
    "200 grams paneer",
    "a pinch of asafoetida",
    "4 cloves garlic",
    "100g besan",
    "2 tbsps ghee",
]
print(f"{'Input':<38} {'qty':>6}  {'unit':<12}  {'name':<28}  {'grams':>7}")
print("-" * 100)
for t in tests:
    p = parse_ingredient(t)
    g = convert_to_grams(p)
    print(f"{t:<38} {p['qty']:>6.3f}  {p['unit']:<12}  {p['name']:<28}  {g:>7.1f}g")

Input                                     qty  unit          name                            grams
----------------------------------------------------------------------------------------------------
2 cups basmati rice                     2.000  cup           basmati rice                    370.0g
1/2 tsp turmeric                        0.500  teaspoon      turmeric                          2.5g
1 1/2 tbsp oil                          1.500  tablespoon    oil                              20.4g
3-4 green chillies                      3.500  piece         green chillies                  280.0g
salt to taste                           0.250  teaspoon      salt                              1.5g
1 medium onion, finely chopped          1.000  medium        onion                            80.0g
200 grams paneer                       200.000  gram          paneer                          200.0g
a pinch of asafoetida                   0.125  teaspoon      asafoetida                        0.6g

## 4. Ingredient Matching & Calculation
Match parsed ingredients to the IFCT database and sum up nutrition.

In [10]:
# ──────────────────────────────────────────────────────────────────────────────
# IFCT Column → Final Dataset Column mapping
# IFCT stores energy in kJ/100g; all others are g or mg or µg per 100g
# ──────────────────────────────────────────────────────────────────────────────
KJ_TO_KCAL = 4.184

IFCT_COL_MAP = {
    'Calories (kcal)':    ('enerc',    lambda v: v / KJ_TO_KCAL),  # kJ → kcal
    'Carbohydrates (g)':  ('choavldf', None),
    'Protein (g)':        ('protcnt',  None),
    'Fats (g)':           ('fatce',    None),
    'Free Sugar (g)':     ('fsugar',   None),
    'Fibre (g)':          ('fibtg',    None),
    'Sodium (mg)':        ('na',       None),
    'Calcium (mg)':       ('ca',       None),
    'Iron (mg)':          ('fe',       None),
    'Vitamin C (mg)':     ('vitc',     None),
    'Folate (µg)':        ('folsum',   None),
}

# Hindi/regional → English aliases for better IFCT matching
ALIASES = {
    'aloo': 'potato', 'pyaz': 'onion', 'tamatar': 'tomato',
    'palak': 'spinach', 'gobi': 'cauliflower', 'gajar': 'carrot',
    'baingan': 'brinjal', 'bhindi': 'okra ladies finger', 'lauki': 'bottle gourd',
    'shimla mirch': 'capsicum', 'methi': 'fenugreek leaves',
    'karela': 'bitter gourd', 'kaddu': 'pumpkin',
    'chana': 'bengal gram chickpea', 'moong': 'green gram',
    'urad': 'black gram', 'masoor': 'red lentil',
    'toor': 'pigeon pea', 'arhar': 'pigeon pea',
    'rajma': 'kidney beans', 'lobia': 'cowpea',
    'haldi': 'turmeric', 'jeera': 'cumin', 'dhaniya': 'coriander',
    'mirchi': 'chilli', 'mirch': 'chilli', 'kali mirch': 'black pepper',
    'hing': 'asafoetida', 'tej patta': 'bay leaf', 'dalchini': 'cinnamon',
    'elaichi': 'cardamom', 'laung': 'clove', 'saunf': 'fennel',
    'chawal': 'rice', 'atta': 'wheat flour', 'maida': 'refined wheat flour',
    'besan': 'bengal gram flour', 'suji': 'semolina', 'rava': 'semolina',
    'poha': 'rice flakes', 'dahi': 'curd', 'ghee': 'ghee',
    'paneer': 'cottage cheese', 'makkhan': 'butter',
    'nariyal': 'coconut', 'imli': 'tamarind',
    'adrak': 'ginger', 'lahsun': 'garlic',
}

# Prepare IFCT lookup
df_ifct['clean_name'] = df_ifct['name'].str.lower().str.strip()
ifct_names = df_ifct['clean_name'].tolist()


def translate_name(name: str) -> str:
    """Translate Hindi ingredient names to English for IFCT matching."""
    name = name.lower().strip()
    for hindi, english in ALIASES.items():
        if hindi in name:
            name = name.replace(hindi, english)
    return name


def get_ifct_nutrition(ing_name: str, grams: float) -> dict | None:
    """
    Fuzzy-match ingredient to IFCT and return scaled nutrition for given grams.
    Returns dict with all 11 nutrition keys, or None if no match.
    """
    if not ing_name or len(ing_name) < 2 or grams <= 0:
        return None

    name = translate_name(ing_name)

    # Try exact match first
    exact = df_ifct[df_ifct['clean_name'] == name]
    if not exact.empty:
        row = exact.iloc[0]
        match_score = 100
    else:
        result = process.extractOne(name, ifct_names, scorer=fuzz.token_set_ratio)
        if not result or result[1] < 55:
            return None
        row = df_ifct[df_ifct['clean_name'] == result[0]].iloc[0]
        match_score = result[1]

    factor = grams / 100.0
    nutrition = {}

    for col_name, (ifct_col, transform) in IFCT_COL_MAP.items():
        try:
            raw_val = float(row[ifct_col]) if pd.notna(row[ifct_col]) else 0.0
        except (KeyError, ValueError, TypeError):
            raw_val = 0.0

        scaled = raw_val * factor
        nutrition[col_name] = round(transform(scaled) if transform else scaled, 4)

    nutrition['_match_score'] = match_score
    nutrition['_matched_to'] = row['name']
    return nutrition


# ──────────────────────────────────────────────────────────────────────────────
# Main Calculation Loop
# ──────────────────────────────────────────────────────────────────────────────
SERVINGS_DEFAULT = 3   # Assume recipe serves 3 people

NUTRI_COLS = list(IFCT_COL_MAP.keys())

final_results = []

for _, row in tqdm(df_matches.iterrows(), total=len(df_matches)):
    ing_str = row['ingredients']

    result_row = {
        'image_class':      row['image_class'],
        'matched_recipe':   row['matched_recipe'],
        'match_score':      row['match_score'],
        'method':           'calculated_ifct',
    }

    if pd.isna(ing_str) or ing_str.strip() == '':
        for col in NUTRI_COLS:
            result_row[col] = None
        result_row['ingredient_coverage'] = 0.0
        result_row['servings_assumed']    = SERVINGS_DEFAULT
        final_results.append(result_row)
        continue

    ing_list = [i.strip() for i in ing_str.split(',') if i.strip()]

    totals       = {col: 0.0 for col in NUTRI_COLS}
    matched_cnt  = 0
    total_cnt    = len(ing_list)

    for ing in ing_list:
        parsed = parse_ingredient(ing)
        grams  = convert_to_grams(parsed)
        nutri  = get_ifct_nutrition(parsed['name'], grams)

        if nutri:
            matched_cnt += 1
            for col in NUTRI_COLS:
                totals[col] += nutri.get(col, 0.0)

    # Divide by servings
    for col in NUTRI_COLS:
        result_row[col] = round(totals[col] / SERVINGS_DEFAULT, 2)

    result_row['ingredient_coverage'] = round(matched_cnt / total_cnt, 2) if total_cnt else 0.0
    result_row['servings_assumed']    = SERVINGS_DEFAULT
    final_results.append(result_row)

df_calculated = pd.DataFrame(final_results)

# Preview
print(f"\n[OK] Calculated nutrition for {len(df_calculated)} image classes")
print(f"   Avg ingredient coverage: {df_calculated['ingredient_coverage'].mean():.1%}")
print(f"   Coverage ≥ 50%: {(df_calculated['ingredient_coverage'] >= 0.5).sum()} classes")
print(f"   Coverage < 50%: {(df_calculated['ingredient_coverage'] < 0.5).sum()} classes")
print()
df_calculated[['image_class', 'Calories (kcal)', 'Protein (g)', 'Carbohydrates (g)', 'Fats (g)', 'ingredient_coverage']].head(10)

  0%|          | 0/94 [00:00<?, ?it/s]


[OK] Calculated nutrition for 94 image classes
   Avg ingredient coverage: 86.4%
   Coverage ≥ 50%: 94 classes
   Coverage < 50%: 0 classes



,image_class,Calories (kcal),Protein (g),Carbohydrates (g),Fats (g),ingredient_coverage
0,achar,273.74,23.61,27.51,5.84,0.89
1,aloo gobi,628.17,20.36,75.73,25.58,0.93
2,aloo methi,453.47,17.82,57.49,14.87,1.00
3,aloo puri,619.76,25.69,79.54,19.07,0.92
4,aloo tikki,301.66,9.33,57.14,3.00,1.00
5,amritsari kulcha,424.22,13.76,73.85,6.73,0.82
6,anarsa,222.43,7.44,27.26,8.51,0.80
7,appam,229.64,13.67,39.09,1.45,1.00
8,baingan bharta,394.46,19.34,48.73,11.87,0.92
9,balushahi,296.18,18.24,21.69,14.77,0.80


## 5. Validate Matches & Save Results
Audit every image_class → matched_recipe pair before saving.  
Flag and **remove** entries where the recipe is fundamentally different from the food class.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5a. Automated Match-Quality Audit
# ─────────────────────────────────────────────────────────────────────────────
# Rules:
# 1. CRITICAL BLACKLIST – recipe is fundamentally a different dish
# 2. FRUIT/VEG as raw item matched to a cooked dish (cake, dessert, curry…)
# 3. Duplicate recipe used for different classes
# ─────────────────────────────────────────────────────────────────────────────

# ── 1. Manual blacklist of clearly wrong matches (found via review) ──────────
BLACKLIST = {
    # image_class : reason
    'boondi':       'Matched to "Boondi Ki Kheer" (dessert), not plain boondi snack',
    'grape':        'Matched to "Grape Wright Recipe" — unclear/suspicious recipe',
    'hot dogs':     'Matched to "Crunchy Carrots In Hot Dogs Buns" — not actual hot dogs',
    'khakhra':      'Matched to "Khakhra Pizza" — pizza, not plain khakhra',
    'kiwi':         'Matched to "Kiwi Guava Spritzer" — drink, not kiwi fruit nutrition',
    'paprika':      'Matched to "Fig Honey & Paprika Preserve" — preserve, not the spice/vegetable',
    'pear':         'Matched to "Upside Down Pear Ginger Cake" — cake, not pear fruit',
    'popcorn':      'Matched to "KFC Style Crispy Chicken Popcorn" — chicken snack, not popcorn',
    'rabri':        'Matched to "Malpua Recipe with Rabri" — malpua dish, not rabri alone',
    'turnip':       'Matched to mutton kofta + turnip curry — not turnip vegetable',
    'waffle':       'Matched to "Waffle Broth - Puppet\'s Yamarinta Crovy" — nonsensical recipe',
    'watermelon':   'Matched to watermelon dessert with nuts/yogurt — not plain watermelon',
}

# ── 2. Heuristic checks for remaining rows ──────────────────────────────────

def heuristic_audit(row):
    """Return a warning string if the match looks suspicious, else ''."""
    cls   = str(row['image_class']).lower().strip()
    rname = str(row['matched_recipe']).lower().strip()
    warnings = []

    # Fruits/vegs matched to cooked/dessert dishes
    raw_items = {'apple', 'banana', 'beetroot', 'cabbage', 'capsicum',
                 'carrots', 'cauliflower', 'grape', 'kiwi', 'pear',
                 'turnip', 'watermelon'}
    cooked_signals = ['cake', 'pie', 'dessert', 'pudding', 'preserve',
                      'spritzer', 'smoothie', 'juice', 'curry', 'korma',
                      'kofta', 'sabzi', 'subzi']
    if cls in raw_items:
        for sig in cooked_signals:
            if sig in rname:
                warnings.append(f'Raw item "{cls}" matched to cooked dish ({sig})')
                break

    # Recipe name contains a completely different food category
    mismatch_pairs = [
        ('pizza', 'khakhra'), ('chicken', 'popcorn'),
        ('malpua', 'rabri'), ('cake', 'pear'),
    ]
    for a, b in mismatch_pairs:
        if cls == b and a in rname and b not in rname.split(a)[0]:
            warnings.append(f'Class "{cls}" but recipe is primarily "{a}"')

    # Duplicate recipe across multiple classes
    # (will be checked dataset-wide below)

    return '; '.join(warnings)


# ── Apply checks ─────────────────────────────────────────────────────────────
df_calculated['_blacklisted'] = df_calculated['image_class'].map(
    lambda x: BLACKLIST.get(x, '')
)
df_calculated['_heuristic_warning'] = df_calculated.apply(heuristic_audit, axis=1)

# Check for duplicate recipes across different classes
recipe_counts = df_calculated.groupby('matched_recipe')['image_class'].apply(list)
dupes = recipe_counts[recipe_counts.apply(len) > 1]
dupe_map = {}
for recipe, classes in dupes.items():
    for cls in classes:
        dupe_map[cls] = f'Recipe "{recipe}" also used for: {[c for c in classes if c != cls]}'
df_calculated['_duplicate_recipe'] = df_calculated['image_class'].map(
    lambda x: dupe_map.get(x, '')
)

# ── Summary ──────────────────────────────────────────────────────────────────
blacklisted = df_calculated[df_calculated['_blacklisted'] != '']
warned      = df_calculated[(df_calculated['_heuristic_warning'] != '') & (df_calculated['_blacklisted'] == '')]
duplicated  = df_calculated[(df_calculated['_duplicate_recipe'] != '') & (df_calculated['_blacklisted'] == '')]

print("=" * 90)
print("MATCH-QUALITY AUDIT RESULTS")
print("=" * 90)

print(f"\n🚫 BLACKLISTED (will be REMOVED): {len(blacklisted)}")
if not blacklisted.empty:
    for _, r in blacklisted.iterrows():
        print(f"   • {r['image_class']:<22} → {r['_blacklisted']}")

print(f"\n[WARN]  HEURISTIC WARNINGS (review needed): {len(warned)}")
if not warned.empty:
    for _, r in warned.iterrows():
        print(f"   • {r['image_class']:<22} → {r['_heuristic_warning']}")

print(f"\n🔄 DUPLICATE RECIPES (same recipe, different class): {len(duplicated)}")
if not duplicated.empty:
    for _, r in duplicated.iterrows():
        print(f"   • {r['image_class']:<22} → {r['_duplicate_recipe']}")

print(f"\n[OK] CLEAN MATCHES: {len(df_calculated) - len(blacklisted) - len(warned)}")
print("=" * 90)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5b. Final Quality Checks & Export (after removing blacklisted matches)
# ─────────────────────────────────────────────────────────────────────────────

# Start from df_calculated, EXCLUDE blacklisted entries first
df_clean = df_calculated[df_calculated['_blacklisted'] == ''].copy()
print(f"After removing {len(BLACKLIST)} blacklisted classes: {len(df_clean)} rows remain")

# Filter out low-quality matches
high_quality_df = df_clean[
    (df_clean['ingredient_coverage'] >= 0.5) &  # At least 50% ingredients matched to IFCT
    (df_clean['match_score'] >= 85)              # High confidence recipe match
].copy()

# Drop suspicious calorie counts (< 50 kcal or > 2000 kcal per serving)
high_quality_df = high_quality_df[
    (high_quality_df['Calories (kcal)'] > 50) & 
    (high_quality_df['Calories (kcal)'] < 2000)
]

# Drop internal audit columns before saving
audit_cols = ['_blacklisted', '_heuristic_warning', '_duplicate_recipe']
high_quality_df = high_quality_df.drop(columns=[c for c in audit_cols if c in high_quality_df.columns])

output_path = '../Dataset/processed/Missing_Image_Classes_Nutrition_Calculated.csv'
high_quality_df.to_csv(output_path, index=False)

print(f"\n[OK] Saved High Quality Dataset to: {output_path}")
print(f"   Initial calculated rows:  {len(df_calculated)}")
print(f"   After blacklist removal:  {len(df_clean)}")
print(f"   Final saved rows:         {len(high_quality_df)}")
print(f"   Columns:                  {list(high_quality_df.columns)}")
print()

print("[STATS] Final Quality Summary:")
print(f"   Avg Match Score:          {high_quality_df['match_score'].mean():.1f}")
print(f"   Avg Ingredient Coverage:  {high_quality_df['ingredient_coverage'].mean():.1%}")

# Sample output
print("\n[INFO] Final Verified Samples:")
high_quality_df[['image_class', 'matched_recipe', 'Calories (kcal)', 'ingredient_coverage']].head(15)

[OK] Saved High Quality Dataset to: ../Dataset/processed/Missing_Image_Classes_Nutrition_Calculated.csv
   Initial calculated rows: 94
   Final saved rows:        94
   Columns:                 ['image_class', 'matched_recipe', 'match_score', 'method', 'Calories (kcal)', 'Carbohydrates (g)', 'Protein (g)', 'Fats (g)', 'Free Sugar (g)', 'Fibre (g)', 'Sodium (mg)', 'Calcium (mg)', 'Iron (mg)', 'Vitamin C (mg)', 'Folate (µg)', 'ingredient_coverage', 'servings_assumed']

[STATS] Final Quality Summary:
   Avg Match Score: 100.0
   Avg Ingredient Coverage: 86.4%

[INFO] Final Verified Samples:


,image_class,matched_recipe,Calories (kcal),ingredient_coverage
0,achar,Singhade Ka Achar Recipe (Water Chestnut Pickle),273.74,0.89
1,aloo gobi,Tandoori Aloo Gobi Kathi Roll Recipe - Spicy Aloo Gobi Frankie,628.17,0.93
2,aloo methi,Methi & Pudina Aloo Sabzi Recipe,453.47,1.00
3,aloo puri,Aloo Masala Puri Recipe,619.76,0.92
4,aloo tikki,Instant Baked Aloo Tikki Recipe,301.66,1.00
5,amritsari kulcha,Tawa Amritsari Kulcha Recipe,424.22,0.82
6,anarsa,Traditional Anarsa Recipe With Curd (Apoopa),222.43,0.80
7,appam,Kerala Style Appam Recipe - Kerala Style Appam Recipe,229.64,1.00
8,baingan bharta,Baingan Bharta Recipe - Brinjal Bharta Recipe,394.46,0.92
9,balushahi,Badusha Recipe | Balushahi Recipe - Sweet Indian Doughnut,296.18,0.80


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 7. Add to Unified Dataset
# ─────────────────────────────────────────────────────────────────────────────
unified_path = '../Dataset/processed/Final_unified_dataset.csv'

try:
    df_unified = pd.read_csv(unified_path)
    print(f"Loaded existing unified dataset with {len(df_unified)} rows.")
except FileNotFoundError:
    print("Unified dataset not found! Please check the path.")
    df_unified = pd.DataFrame()

if not df_unified.empty:
    print("Unified Columns:", df_unified.columns.tolist())
    print()

    # ── Join high_quality_df back with df_matches to recover ingredients & instructions ──
    merge_cols = ['image_class', 'ingredients', 'instructions']
    df_to_add = high_quality_df.merge(
        df_matches[merge_cols],
        on='image_class',
        how='left',
        suffixes=('', '_from_matches')
    )

    # ── Build new rows aligned to unified schema ─────────────────────────────
    UNIFIED_COLS = df_unified.columns.tolist()
    NUTRI_COLS_SHARED = [
        'Calories (kcal)', 'Carbohydrates (g)', 'Protein (g)', 'Fats (g)',
        'Free Sugar (g)', 'Fibre (g)', 'Sodium (mg)', 'Calcium (mg)',
        'Iron (mg)', 'Vitamin C (mg)', 'Folate (µg)',
    ]

    new_rows = []
    for _, row in df_to_add.iterrows():
        new_row = {col: np.nan for col in UNIFIED_COLS}

        # ── Identity columns ──
        new_row['recipe_original']       = row['matched_recipe']
        new_row['final_food_name']       = row['image_class'].replace('_', ' ').strip().title()
        new_row['best_match_clean']      = row['image_class'].replace('_', ' ').strip().lower()
        new_row['matched_original_name'] = row['matched_recipe']

        # ── Recipe text columns ──
        new_row['Cleaned-Ingredients']    = row.get('ingredients', np.nan)
        new_row['TranslatedIngredients']  = row.get('ingredients', np.nan)
        new_row['TranslatedInstructions'] = row.get('instructions', np.nan)

        # ── Matching score columns ──
        new_row['composite_score']   = row['match_score']
        new_row['token_set_score']   = row['match_score']
        new_row['wratio_score']      = np.nan
        new_row['food_token_score']  = np.nan
        new_row['negative_penalty']  = 0
        new_row['token_overlap']     = np.nan

        # ── Nutrition columns (names already match) ──
        for col in NUTRI_COLS_SHARED:
            if col in UNIFIED_COLS:
                new_row[col] = row[col]

        new_rows.append(new_row)

    df_new_data = pd.DataFrame(new_rows, columns=UNIFIED_COLS)

    # ── Deduplicate: drop any classes already present in unified ─────────────
    existing_names = set(df_unified['final_food_name'].str.lower().str.strip().dropna())
    before_dedup = len(df_new_data)
    df_new_data = df_new_data[
        ~df_new_data['final_food_name'].str.lower().str.strip().isin(existing_names)
    ]
    dupes_dropped = before_dedup - len(df_new_data)
    if dupes_dropped:
        print(f"[WARN]  Skipped {dupes_dropped} classes already present in unified dataset.")

    if df_new_data.empty:
        print("ℹ️  No new rows to add (all classes already exist).")
    else:
        # ── Append & save ────────────────────────────────────────────────────
        df_final = pd.concat([df_unified, df_new_data], ignore_index=True)
        df_final.to_csv(unified_path, index=False)

        print(f"\n[OK] Added {len(df_new_data)} new rows to Final Unified Dataset.")
        print(f"   Previous rows: {len(df_unified)}")
        print(f"   Total rows now: {len(df_final)}")

        # ── Quick sanity check ───────────────────────────────────────────────
        print("\n[STATS] Newly added samples:")
        display(
            df_final.tail(len(df_new_data))[
                ['final_food_name', 'recipe_original', 'Calories (kcal)',
                 'Protein (g)', 'Fats (g)', 'Carbohydrates (g)']
            ].head(15)
        )
else:
    print("Skipping merge — unified dataset was not found.")